# agent-fault-lab walkthrough

A clean eval cannot tell these four policies apart. Fault injection can. Everything here is seeded and needs no API keys.

In [1]:
from agentfaultlab import NONE, MEDIUM, HIGH, POLICIES, run_experiment, run_grid, summarize
from agentfaultlab.run import to_markdown
print(to_markdown(run_grid([NONE], tasks=100, trials=16)))

| faults | policy | task success | duplicate charges | silent failures | unconfirmed | pass^8 |
|---|---|---|---|---|---|---|
| none | naive-retry | 100.0% | 0.0% (0.0% to 0.2%) | 0.0% | 0.0% | 100.0% |
| none | verify-before-retry | 100.0% | 0.0% (0.0% to 0.2%) | 0.0% | 0.0% | 100.0% |
| none | idempotency-key | 100.0% | 0.0% (0.0% to 0.2%) | 0.0% | 0.0% | 100.0% |
| none | idempotency-key+verify | 100.0% | 0.0% (0.0% to 0.2%) | 0.0% | 0.0% | 100.0% |



Every policy scores 100% with no faults, including pass^8. Now inject the three failure modes at the medium level.

In [2]:
print(to_markdown(run_grid([MEDIUM], tasks=100, trials=16)))

| faults | policy | task success | duplicate charges | silent failures | unconfirmed | pass^8 |
|---|---|---|---|---|---|---|
| medium | naive-retry | 81.4% | 9.6% (8.2% to 11.1%) | 17.4% | 0.0% | 20.1% |
| medium | verify-before-retry | 88.8% | 1.7% (1.2% to 2.4%) | 11.2% | 0.0% | 39.6% |
| medium | idempotency-key | 89.3% | 0.0% (0.0% to 0.2%) | 9.7% | 0.0% | 37.5% |
| medium | idempotency-key+verify | 99.8% | 0.0% (0.0% to 0.2%) | 0.0% | 0.2% | 98.5% |



Where does a naive retry go wrong? Look at individual episodes, not just rates.

In [3]:
from agentfaultlab import NaiveRetry
eps = run_experiment(NaiveRetry(), MEDIUM, tasks=20, trials=4)
bad = [e for e in eps if e.silent_failure][:5]
for e in bad:
    print(f'task {e.task} trial {e.trial}: reported {e.report!r}, real state: {e.count} charge(s), fulfilled={e.fulfilled}')

task 0 trial 0: reported 'success', real state: 1 charge(s), fulfilled=False
task 1 trial 2: reported 'success', real state: 1 charge(s), fulfilled=False
task 2 trial 1: reported 'success', real state: 1 charge(s), fulfilled=False
task 2 trial 2: reported 'success', real state: 1 charge(s), fulfilled=False
task 4 trial 0: reported 'success', real state: 2 charge(s), fulfilled=True


## Plug in your own policy

Any object with a `name` and `run(ledger, invoice, amount)` that returns `success`, `failure` or `unconfirmed`. Here is a policy that retries only once.

In [4]:
from agentfaultlab import ToolTimeout

class RetryOnce:
    name = 'retry-once'
    def run(self, ledger, invoice, amount):
        for _ in range(2):
            try:
                ledger.charge(invoice, amount)
                return 'success'
            except ToolTimeout:
                pass
        return 'failure'

print(summarize(run_experiment(RetryOnce(), MEDIUM, tasks=100, trials=16), k=8))

{'episodes': 1600, 'task_success': 0.805625, 'duplicate_action_rate': 0.083125, 'duplicate_ci': (0.07057540127576642, 0.09767163758393144), 'silent_failure_rate': 0.150625, 'silent_ci': (0.1339363861877859, 0.1689872940559489), 'unconfirmed_rate': 0.0, 'pass_hat_8': 0.1841756021756022}
